# Hermes MSGraph — Exemplos e Testes

Este notebook mostra como usar cada função da biblioteca **hermes_msgraph**, organizadas por serviço:

- Setup e autenticação
- `UsersService` — usuários e licenças
- `MailboxFolderService` — pastas de caixa de correio
- `EmailService` — envio e leitura de e-mails
- `PlannerService` — planos e tarefas
- `TeamsService` — times, canais, chats e mensagens

> **Antes de rodar:** preencha suas credenciais na célula de configuração abaixo. Nenhuma credencial real deve ser commitada neste notebook — use variáveis de ambiente ou um `.env` local (veja a célula de setup).
>
> Todas as chamadas abaixo batem na API real do Microsoft Graph. Rode célula a célula e ajuste os valores de exemplo (e-mails, IDs) para os do seu tenant.


## 1. Instalação

Se ainda não instalou a biblioteca:
```bash
pip install git+https://github.com/celoramalho/hermes_msgraph.git
```
ou, em modo de desenvolvimento local (a partir da raiz do repositório):
```bash
pip install -e .
```


In [ ]:
# Descomente se precisar instalar a partir do repositório local
# %pip install -e ..


## 2. Configuração e autenticação

A biblioteca usa autenticação **app-only** (client credentials) do Azure AD / Entra ID.
Você precisa de `client_id`, `client_secret` e `tenant_id` de uma aplicação registrada no Azure Portal,
com as permissões de aplicativo (application permissions) necessárias para os recursos que for testar.

Recomenda-se carregar essas credenciais de variáveis de ambiente, nunca hard-coded no notebook.


In [ ]:
import os

CLIENT_ID = os.getenv("HERMES_CLIENT_ID", "SEU_CLIENT_ID")
CLIENT_SECRET = os.getenv("HERMES_CLIENT_SECRET", "SEU_CLIENT_SECRET")
TENANT_ID = os.getenv("HERMES_TENANT_ID", "SEU_TENANT_ID")

# E-mails/IDs de exemplo usados nas células abaixo — ajuste para o seu tenant
TEST_MAILBOX = os.getenv("HERMES_TEST_MAILBOX", "usuario@suaempresa.com")
TEST_RECIPIENT = os.getenv("HERMES_TEST_RECIPIENT", "destinatario@suaempresa.com")


In [ ]:
from hermes_msgraph import HermesMSGraph
from hermes_msgraph.exceptions import HermesMSGraphError

hermes = HermesMSGraph(
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET,
    tenant_id=TENANT_ID,
)
print("Conectado com sucesso!")


In [ ]:
# Verifica quais permissões (escopos) o token da aplicação realmente possui.
# Útil para depurar erros 403 Forbidden.
hermes.list_msgraph_permissions()


## 3. UsersService — Usuários e licenças

Funções para consultar usuários do tenant, buscar por nome/e-mail e listar licenças (SKUs) disponíveis.


### `get_user_id_by_email`
Retorna o ID (GUID) de um usuário a partir do endereço de e-mail. Lança `HermesMSGraphError` se o usuário não for encontrado.


In [ ]:
user_id = hermes.get_user_id_by_email(TEST_MAILBOX)
user_id


### `get_all_users`
Lista todos os usuários do tenant (membros, não guests). Use `data="simple"` para um resumo (nome, cargo, e-mail, localização)
ou `data="all"` para o payload completo do Graph (inclui licenças, plans, departamento, etc).


In [ ]:
usuarios_simples = hermes.get_all_users(data="simple")
print(f"Total de usuários: {len(usuarios_simples)}")
usuarios_simples[:5]


In [ ]:
usuarios_completos = hermes.get_all_users(data="all")
usuarios_completos[0] if usuarios_completos else None


### `search_from_mailboxes`
Busca usuários pelo nome de exibição usando `$search` (busca aproximada, não é um filtro exato).


In [ ]:
resultados = hermes.search_from_mailboxes("Silva")
resultados


### `get_tenant_licenses`
Lista as licenças (SKUs) do tenant, já com um nome amigável (`friendlyName`) quando reconhecido.
SKUs desconhecidos aparecem como `"Desconhecido"` em vez de causar erro.


In [ ]:
licencas = hermes.get_tenant_licenses()
licencas


## 4. MailboxFolderService — Pastas de caixa de correio


### `list_mailbox_folders` / `get_mailbox_folders`
Lista as pastas de uma caixa de correio. A segunda retorna como `pandas.DataFrame`.


In [ ]:
pastas = hermes.list_mailbox_folders(TEST_MAILBOX)
pastas[:5]


In [ ]:
df_pastas = hermes.get_mailbox_folders(TEST_MAILBOX)
df_pastas.head()


### `get_folder_id` / `validate_folder_id`
Busca o ID de uma pasta pelo nome de exibição, e valida se um ID de pasta existe na caixa.


In [ ]:
folder_id = hermes.get_folder_id(TEST_MAILBOX, "Inbox")
folder_id


In [ ]:
hermes.validate_folder_id(TEST_MAILBOX, folder_id)


## 5. EmailService — Envio e leitura de e-mails


### `send_email`
Envia um e-mail, com suporte a CC, anexos (paths locais) e envio agendado (`delay` em segundos).


In [ ]:
hermes.send_email(
    sender_mail=TEST_MAILBOX,
    subject="Teste Hermes MSGraph",
    body="Este é um e-mail de teste enviado pela biblioteca hermes_msgraph.",
    to_address=TEST_RECIPIENT,
    body_type="Text",
)


### `get_emails`
Lista e-mails de uma caixa com filtros opcionais: assunto (suporta `*prefixo`, `sufixo*`, `*contém*`), pasta, remetente,
quantidade, presença de anexos e intervalo de datas. Pode retornar `list` ou `pandas.DataFrame` via `format=`.


In [ ]:
emails = hermes.get_emails(TEST_MAILBOX, n_of_messages=5)
emails


In [ ]:
import pandas as pd

df_emails = hermes.get_emails(TEST_MAILBOX, n_of_messages=5, format=pd.DataFrame)
df_emails.head()


### `list_email_attachments` / `download_attachment`
Lista os anexos de um e-mail específico e baixa um anexo para um arquivo local.


In [ ]:
if emails:
    primeiro_email_id = emails[0]["id"]
    anexos = hermes.list_email_attachments(primeiro_email_id, TEST_MAILBOX)
    anexos


### `forward_email_by_id`
Encaminha um e-mail existente para outro destinatário, com comentário opcional.


In [ ]:
if emails:
    hermes.forward_email_by_id(
        email_id=emails[0]["id"],
        mailbox_address=TEST_MAILBOX,
        to_address=TEST_RECIPIENT,
        comment="Encaminhando via hermes_msgraph",
    )


### `list_sharepoint_sites`
Lista os sites raiz do SharePoint do tenant.


In [ ]:
sites = hermes.list_sharepoint_sites()
sites


## 6. PlannerService — Planos e tarefas

Requer que o usuário/grupo tenha planos do Microsoft Planner associados.


In [ ]:
GROUP_ID = os.getenv("HERMES_TEST_GROUP_ID", "SEU_GROUP_ID")

planos = hermes.list_plans_by_group_id(GROUP_ID, data="processed")
planos


In [ ]:
planos_visiveis = hermes.list_visible_plans_by_user_id(user_id)
planos_visiveis


In [ ]:
tarefas_usuario = hermes.list_tasks_by_user_id(user_id)
tarefas_usuario


In [ ]:
if planos:
    tarefas_plano = hermes.list_tasks_by_plan_id(planos[0]["id"])
    tarefas_plano


## 7. TeamsService — Times, canais e chats

Requer permissões de aplicativo: `Team.ReadBasic.All`, `Channel.ReadBasic.All`, `ChannelMessage.Read.All`,
`Chat.Read.All`, `ChatMember.Read.All`.


### `list_all_teams` / `list_joined_teams_by_user_id`


In [ ]:
times = hermes.list_all_teams()
times[:5]


In [ ]:
times_do_usuario = hermes.list_joined_teams_by_user_id(user_id)
times_do_usuario


### `list_channels_by_team_id` / `get_channel_by_id`
`include_private=True` usa o endpoint beta para incluir canais privados/compartilhados a que o app tem acesso.


In [ ]:
if times:
    TEAM_ID = times[0]["id"]
    canais = hermes.list_channels_by_team_id(TEAM_ID)
    canais


In [ ]:
if times and canais:
    CHANNEL_ID = canais[0]["id"]
    canal = hermes.get_channel_by_id(TEAM_ID, CHANNEL_ID)
    canal


### `list_channel_messages` / `list_channel_message_replies`
`include_replies=True` (padrão) aninha as respostas de cada mensagem sob a chave `"replies"`.


In [ ]:
if times and canais:
    mensagens_canal = hermes.list_channel_messages(TEAM_ID, CHANNEL_ID)
    mensagens_canal[:3]


### `delta_channel_messages`
Sincronização incremental: guarde o `delta_link` retornado e passe-o na próxima chamada para buscar só as mudanças.


In [ ]:
if times and canais:
    resultado_delta = hermes.delta_channel_messages(TEAM_ID, CHANNEL_ID)
    novo_delta_link = resultado_delta["delta_link"]
    print(f"Mensagens: {len(resultado_delta['messages'])}")
    print(f"Delta link para a próxima sincronização: {novo_delta_link}")


### `list_chats_by_user_id` / `get_chat_by_id` / `list_chat_members` / `list_chat_messages`


In [ ]:
chats = hermes.list_chats_by_user_id(user_id)
chats[:3]


In [ ]:
if chats:
    CHAT_ID = chats[0]["id"]
    chat = hermes.get_chat_by_id(CHAT_ID)
    membros = hermes.list_chat_members(CHAT_ID)
    mensagens_chat = hermes.list_chat_messages(CHAT_ID)
    print(chat)
    print(membros)
    print(mensagens_chat[:3])


### `delta_chat_messages`
Mesmo padrão de sincronização incremental usado em `delta_channel_messages`, mas para chats.


In [ ]:
if chats:
    resultado_delta_chat = hermes.delta_chat_messages(CHAT_ID)
    print(f"Mensagens: {len(resultado_delta_chat['messages'])}")
    print(f"Delta link: {resultado_delta_chat['delta_link']}")


### `download_hosted_content`
Baixa conteúdo binário hospedado referenciado numa mensagem (ex.: imagens inline em `hostedContents/{id}/$value`).


In [ ]:
# Exemplo: pegar a URL de um hostedContent a partir de uma mensagem de chat/canal que tenha anexos inline
# hosted_content_url = mensagens_chat[0]["attachments"][0]["contentUrl"]
# hermes.download_hosted_content(hosted_content_url, "anexo_baixado.png")


## 8. Tratamento de erros

Todos os métodos lançam `HermesMSGraphError` quando a API do Graph retorna um erro (status != 200/202).


In [ ]:
try:
    hermes.get_user_id_by_email("usuario-que-nao-existe@suaempresa.com")
except HermesMSGraphError as e:
    print(f"Erro esperado: {e}")


## Observações finais

- `add_user_to_shared_mailbox` **não está implementada**: a Microsoft Graph API não expõe um endpoint equivalente a
  `Add-MailboxPermission`/`Add-RecipientPermission` do Exchange Online PowerShell. Conceder acesso a uma shared mailbox
  hoje só é possível via Exchange Online PowerShell ou pelo Admin Center — fora do escopo desta biblioteca.
- Chamadas de paginação (`get_all_users`, `list_all_teams`, etc.) seguem `@odata.nextLink` automaticamente.
- Para tenants grandes, prefira `data="simple"` em `get_all_users` quando não precisar do payload completo.
